In [0]:
spark

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/github_data/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/github_data/repositories.json,repositories.json,641853,1784530778000


In [0]:
df = spark.read.json("/Volumes/workspace/default/github_data/repositories.json")

my file is json array but spark uses json lines. need to modify how i am loaing the data into the spark dataframe

In [0]:
df = spark.read.option("multiLine", "true").json("/Volumes/workspace/default/github_data/repositories.json")

In [0]:
df.show()

+--------------------+-----+--------------------+----------+--------+--------------------+--------------------+-----------+--------------------+------+--------------------+--------------------+--------+
|          created_at|forks|           full_name|        id|language|             license|                name|open_issues|               owner| stars|          updated_at|                 url|watchers|
+--------------------+-----+--------------------+----------+--------+--------------------+--------------------+-----------+--------------------+------+--------------------+--------------------+--------+
|2016-03-20T23:49:42Z|49663|public-apis/publi...|  54346799|  Python|         MIT License|         public-apis|       1569|         public-apis|451383|2026-07-19T20:25:20Z|https://github.co...|  451383|
|2013-10-11T06:50:37Z|66552|EbookFoundation/f...|  13491895|  Python|Creative Commons ...|free-programming-...|         71|     EbookFoundation|392490|2026-07-19T19:51:43Z|https://github.c

In [0]:
df.printSchema()

root
 |-- created_at: string (nullable = true)
 |-- forks: long (nullable = true)
 |-- full_name: string (nullable = true)
 |-- id: long (nullable = true)
 |-- language: string (nullable = true)
 |-- license: string (nullable = true)
 |-- name: string (nullable = true)
 |-- open_issues: long (nullable = true)
 |-- owner: string (nullable = true)
 |-- stars: long (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- url: string (nullable = true)
 |-- watchers: long (nullable = true)



In [0]:
df.show(5, truncate=False)

+--------------------+-----+------------------------------------------+--------+--------+----------------------------------------------+----------------------+-----------+-------------------+------+--------------------+-------------------------------------------------------------+--------+
|created_at          |forks|full_name                                 |id      |language|license                                       |name                  |open_issues|owner              |stars |updated_at          |url                                                          |watchers|
+--------------------+-----+------------------------------------------+--------+--------+----------------------------------------------+----------------------+-----------+-------------------+------+--------------------+-------------------------------------------------------------+--------+
|2016-03-20T23:49:42Z|49663|public-apis/public-apis                   |54346799|Python  |MIT License                           

In [0]:
df.count()

1400

In [0]:
df.describe().show()

+-------+--------------------+-----------------+--------------------+--------------------+----------+------------------+------+-----------------+----------+-----------------+--------------------+--------------------+-----------------+
|summary|          created_at|            forks|           full_name|                  id|  language|           license|  name|      open_issues|     owner|            stars|          updated_at|                 url|         watchers|
+-------+--------------------+-----------------+--------------------+--------------------+----------+------------------+------+-----------------+----------+-----------------+--------------------+--------------------+-----------------+
|  count|                1400|             1400|                1400|                1400|      1400|              1324|  1400|             1400|      1400|             1400|                1400|                1400|             1400|
|   mean|                NULL|6122.685714285714|            

In [0]:
df.select("license").distinct().show(20, truncate=False)

+----------------------------------------------------------+
|license                                                   |
+----------------------------------------------------------+
|GNU General Public License v2.0                           |
|Mozilla Public License 2.0                                |
|The Unlicense                                             |
|BSD 3-Clause "New" or "Revised" License                   |
|Creative Commons Zero v1.0 Universal                      |
|GNU Lesser General Public License v3.0                    |
|GNU Lesser General Public License v2.1                    |
|Other                                                     |
|GNU Affero General Public License v3.0                    |
|Open Software License 3.0                                 |
|Creative Commons Attribution 4.0 International            |
|Boost Software License 1.0                                |
|zlib License                                              |
|Microsoft Public Licens

transformation...

In [0]:
from pyspark.sql.functions import col, when, to_timestamp

In [0]:
clean_df = (
    df
    .withColumn(
        "license",
        when(col("license").isNull(), "No License")
        .otherwise(col("license"))
    )
)

any null values for license should now say "no license". lets check

In [0]:
clean_df.select("license").distinct().show(20, truncate=False)

+----------------------------------------------------------+
|license                                                   |
+----------------------------------------------------------+
|GNU General Public License v2.0                           |
|Mozilla Public License 2.0                                |
|The Unlicense                                             |
|BSD 3-Clause "New" or "Revised" License                   |
|Creative Commons Zero v1.0 Universal                      |
|GNU Lesser General Public License v3.0                    |
|GNU Lesser General Public License v2.1                    |
|Other                                                     |
|GNU Affero General Public License v3.0                    |
|Open Software License 3.0                                 |
|Creative Commons Attribution 4.0 International            |
|No License                                                |
|Boost Software License 1.0                                |
|zlib License           

now time to convert the dates 

In [0]:
from pyspark.sql.functions import to_timestamp

In [0]:
clean_df = (
    clean_df
    .withColumn("created_at", to_timestamp("created_at"))
    .withColumn("updated_at", to_timestamp("updated_at"))
)

In [0]:
clean_df.printSchema()

root
 |-- created_at: timestamp (nullable = true)
 |-- forks: long (nullable = true)
 |-- full_name: string (nullable = true)
 |-- id: long (nullable = true)
 |-- language: string (nullable = true)
 |-- license: string (nullable = true)
 |-- name: string (nullable = true)
 |-- open_issues: long (nullable = true)
 |-- owner: string (nullable = true)
 |-- stars: long (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- url: string (nullable = true)
 |-- watchers: long (nullable = true)



good now time time for feature engineering

In [0]:
from pyspark.sql.functions import col, current_timestamp, datediff, round

In [0]:
clean_df = clean_df.withColumn(
    "repo_age_years",
    round(
        datediff(current_timestamp(), col("created_at")) / 365,
        1
    )
)

In [0]:
clean_df = clean_df.withColumn(
    "popularity_score",
    col("stars") + col("forks")
)

i just created 2 fields that might be interesting for analytics: first is repo' age, 2nd one is popularity score field

In [0]:
clean_df.select(
    "name",
    "language",
    "stars",
    "forks",
    "repo_age_years",
    "popularity_score"
).show(10, truncate=False)

+----------------------+--------+------+-----+--------------+----------------+
|name                  |language|stars |forks|repo_age_years|popularity_score|
+----------------------+--------+------+-----+--------------+----------------+
|public-apis           |Python  |451383|49663|10.3          |501046          |
|free-programming-books|Python  |392490|66552|12.8          |459042          |
|system-design-primer  |Python  |358271|57276|9.4           |415547          |
|awesome-python        |Python  |309118|28348|12.1          |337466          |
|project-based-learning|Python  |274082|35329|9.3           |309411          |
|Python                |Python  |222896|50871|10.0          |273767          |
|hermes-agent          |Python  |217208|40878|1.0           |258086          |
|AutoGPT               |Python  |185616|46073|3.3           |231689          |
|yt-dlp                |Python  |178910|15216|5.7           |194126          |
|markitdown            |Python  |167300|12031|1.7   

since we have our silver dataset, need to do one more thing.... JSON is great for APIs and web development. But for data engineering at scale, it is is slow, expensive, and not efficient. while something like parquet is designed specifically for analytics.

In [0]:
clean_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/github_data/silver/repositories.parquet"
)

In [0]:
silver_df = spark.read.parquet(
    "/Volumes/workspace/default/github_data/silver/repositories.parquet"
)

In [0]:
silver_df.show(5)

+-------------------+-----+--------------------+--------+--------+--------------------+--------------------+-----------+-------------------+------+-------------------+--------------------+--------+--------------+----------------+
|         created_at|forks|           full_name|      id|language|             license|                name|open_issues|              owner| stars|         updated_at|                 url|watchers|repo_age_years|popularity_score|
+-------------------+-----+--------------------+--------+--------+--------------------+--------------------+-----------+-------------------+------+-------------------+--------------------+--------+--------------+----------------+
|2016-03-20 23:49:42|49663|public-apis/publi...|54346799|  Python|         MIT License|         public-apis|       1569|        public-apis|451383|2026-07-19 20:25:20|https://github.co...|  451383|          10.3|          501046|
|2013-10-11 06:50:37|66552|EbookFoundation/f...|13491895|  Python|Creative Commo

now need to have the parquet data as a table otherwise sql cant query it

In [0]:
silver_df.createOrReplaceTempView("repositories")

now i can run sql queries in the spark session... NOTE THAT THIS TABLE IS TEMPORARY 

In [0]:
spark.sql("""
SELECT *
FROM repositories
LIMIT 5
""").show()

+-------------------+-----+--------------------+--------+--------+--------------------+--------------------+-----------+-------------------+------+-------------------+--------------------+--------+--------------+----------------+
|         created_at|forks|           full_name|      id|language|             license|                name|open_issues|              owner| stars|         updated_at|                 url|watchers|repo_age_years|popularity_score|
+-------------------+-----+--------------------+--------+--------+--------------------+--------------------+-----------+-------------------+------+-------------------+--------------------+--------+--------------+----------------+
|2016-03-20 23:49:42|49663|public-apis/publi...|54346799|  Python|         MIT License|         public-apis|       1569|        public-apis|451383|2026-07-19 20:25:20|https://github.co...|  451383|          10.3|          501046|
|2013-10-11 06:50:37|66552|EbookFoundation/f...|13491895|  Python|Creative Commo

1st question: which languages dominate the most popular repositories?

In [0]:
language_popularity = spark.sql("""
SELECT
    language,
    COUNT(*) AS repository_count,
    ROUND(AVG(stars),2) AS avg_stars,
    ROUND(AVG(popularity_score),2) AS avg_popularity
FROM repositories
GROUP BY language
ORDER BY avg_popularity DESC
""")

language_popularity.show(20)

+----------+----------------+---------+--------------+
|  language|repository_count|avg_stars|avg_popularity|
+----------+----------------+---------+--------------+
|    Python|             100|102174.95|      118705.1|
|TypeScript|             100| 84966.87|      97127.54|
|JavaScript|             100| 65382.67|      75599.53|
|        Go|             100| 48584.99|      54248.19|
|      Rust|             100| 46110.47|      50473.26|
|       C++|             100| 39571.29|      46678.15|
|      Java|             100| 35879.55|       45436.9|
|         C|             100| 29721.91|      35196.66|
|        C#|             100| 19166.23|      22015.52|
|     Swift|             100| 17979.15|      19764.16|
|    Kotlin|             100| 16826.45|      19299.33|
|       PHP|             100| 16007.53|       18927.5|
|      Ruby|             100| 15368.46|      18117.78|
|      Dart|             100|  13032.0|       14900.5|
+----------+----------------+---------+--------------+



2nd question: what are the most influential repos?

In [0]:
top_repos = spark.sql("""
SELECT
    name,
    owner,
    language,
    stars,
    forks,
    popularity_score,
    url
FROM repositories
ORDER BY popularity_score DESC
LIMIT 10
""")

top_repos.show(truncate=False)

+----------------------+-------------------+----------+------+------+----------------+-------------------------------------------------------------+
|name                  |owner              |language  |stars |forks |popularity_score|url                                                          |
+----------------------+-------------------+----------+------+------+----------------+-------------------------------------------------------------+
|public-apis           |public-apis        |Python    |451383|49663 |501046          |https://github.com/public-apis/public-apis                   |
|freeCodeCamp          |freeCodeCamp       |TypeScript|452087|45595 |497682          |https://github.com/freeCodeCamp/freeCodeCamp                 |
|openclaw              |openclaw           |TypeScript|383486|80555 |464041          |https://github.com/openclaw/openclaw                         |
|free-programming-books|EbookFoundation    |Python    |392490|66552 |459042          |https://github.com/E

3rd question: which licenses are the most common among the popular repos?

In [0]:
license_analysis = spark.sql("""
SELECT
    license,
    COUNT(*) AS repository_count,
    ROUND(AVG(stars),2) AS avg_stars
FROM repositories
GROUP BY license
ORDER BY repository_count DESC
""")

license_analysis.show(20)

+--------------------+----------------+---------+
|             license|repository_count|avg_stars|
+--------------------+----------------+---------+
|         MIT License|             511| 41727.17|
|  Apache License 2.0|             278| 37281.06|
|               Other|             210| 45217.88|
|GNU General Publi...|             121| 26831.24|
|          No License|              76| 33025.96|
|GNU Affero Genera...|              63| 39343.29|
|BSD 3-Clause "New...|              44| 44130.73|
|GNU General Publi...|              22| 28754.77|
|BSD 2-Clause "Sim...|              16|  25083.0|
|Mozilla Public Li...|              12| 32156.58|
|       The Unlicense|               8| 77886.13|
|Creative Commons ...|               7| 48769.14|
|GNU Lesser Genera...|               6| 24297.33|
|Creative Commons ...|               5| 120487.2|
|        zlib License|               4|  20728.0|
|Creative Commons ...|               4| 36948.75|
|GNU Lesser Genera...|               4|  20980.5|


now i need to save these dataframes containing this information I found

In [0]:
language_popularity.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/github_data/gold/language_popularity"
)

In [0]:
top_repos.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/github_data/gold/top_repositories"
)

In [0]:
license_analysis.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/github_data/gold/license_analysis"
)

right now this data exists as folders, should also create tables for sql queries in an sql editor

In [0]:
language_popularity = spark.read.parquet(
    "/Volumes/workspace/default/github_data/gold/language_popularity"
)

language_popularity.write \
    .mode("overwrite") \
    .saveAsTable("language_popularity")

In [0]:
top_repos = spark.read.parquet(
    "/Volumes/workspace/default/github_data/gold/top_repositories"
)

top_repos.write \
    .mode("overwrite") \
    .saveAsTable("top_repositories")

In [0]:
license_analysis = spark.read.parquet(
    "/Volumes/workspace/default/github_data/gold/license_analysis"
)

license_analysis.write \
    .mode("overwrite") \
    .saveAsTable("license_analysis")

lets check if we have the tables now

In [0]:
spark.sql("SHOW TABLES").show()

+--------+-------------------+-----------+
|database|          tableName|isTemporary|
+--------+-------------------+-----------+
| default|language_popularity|      false|
| default|   license_analysis|      false|
| default|   top_repositories|      false|
+--------+-------------------+-----------+

